In [ ]:
from langchain_ollama import ChatOllama
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Literal,Annotated,operator
from langchain_core.messages import SystemMessage,HumanMessage
from pydantic import BaseModel,Field

In [ ]:
generatorLLM=ChatOllama(model="llama3")
evaluatorLLM=ChatOllama(model="llama3")
optimizerLLM=ChatOllama(model="llama3")

In [ ]:
class TweetEvaluation(BaseModel):
    evaluation : Literal["approved","Needs_Improvement"] = Field(description="Final Evaluation Result")
    feedBack : str=Field(description="Feedback for the tweet")
    
structuredEvaluatorLLM=evaluatorLLM.with_structured_output(TweetEvaluation)    

In [ ]:
class TweetState(TypedDict):
    topic   :   str
    tweet   :   str
    evaluation  :   Literal["approved","Needs_Improvement"]
    feedback    :   str
    iteration   :   int
    maxIteration    :   int
    
    
    tweetHistory    :   Annotated[list[str],operator.add]
    feedBackHistory :   Annotated[list[str],operator.add]
    
    

In [ ]:
def generateTweet(state:TweetState):
    #   Define Prompt
    
    messages=[
        SystemMessage(content="You are a funny clever twitter influencer"),
        HumanMessage(content=f"""Write a short,Hilarious,Original tweet on the topic :"{state['topic']}"
                     Rules : 
                     -Do not use question-answer format
                     -Maximum 300 characters
                     -Use Obervational humor,irony,sarcasm or cultural references
                     -Think in meme,logic,punchlines or relateable tasks
                     -Use Simple day to day English language
                     """)
    ]
    
    #   Send prompt to LLM
    response=generatorLLM.invoke(messages)
    
    #   Return response
    return {"tweet":response, "tweetHistory":[response]}

In [ ]:
def evaluateTweet(state:TweetState):
    #   Prompt
    messages=[
        SystemMessage(content="You are a ruthless,no-laugh-given Twitter critic,\
                              You evaluate tweets based on Humor,Originality,Virality and Tweet format"),
        
        HumanMessage(content=f"""
                     Evaluate the following tweet\n
                     Tweet : {state['tweet']}"
                     Use below criteria to evaluate the Tweet:
                     -  Originality :   Is this fresh , Have you seen it before 100 times?
                     -  Humor   :  Did it actually made you smile,laugh or chukle
                     -  Punchiness  :   Is it short,sharp and scroll stopping
                     -  Virality Potential  : Would people retweet or share it?
                     -  Format  :   Is it well-formed tweet(not a setup-punchline joke,not a Q&A joke,Under 300 characters)
                     ### Respond ONLY in structured format:
                     - evaluation: "approved" or "needs_improvement"  
                     - feedback: One paragraph explaining the strengths and weaknesses 
                    """
                     )
    ]
    
    response=structuredEvaluatorLLM.invoke(messages)
    
    return {"evaluation":response.evaluation,
            'feedback':response.feedBack,
            'feedBackHistory':[response.feedBack]
            }

In [ ]:
def optimizeTweet(state:TweetState):
    messages=[
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback"),
        HumanMessage(content=f"""
                     Improve the tweet based on this feedback
                     Feedback : "{state['feedBack']}"
                     
                     Topic  :{state['topic']}
                     Original Tweet : {state['tweet']}
                     ReWrite it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
                     """
                    )
    
    ]
    
    response=optimizerLLM.invoke(messages).content
    
    iteration=state['iteration']+1
    
    return {
        'tweet' : response,
        'iteration' : iteration,
        'tweetHiistory' :[response]
    }

In [ ]:
def routeEvaluation(state:TweetState):
    if state['evaluation']=="approved" or state['iteration'] >= state['maxIteration']:
        return "approved"
    else:
        return "Needs_Improvement"

In [ ]:
graph=StateGraph(TweetState)


graph.add_node("generate",generateTweet)

graph.add_node("evaluate",evaluateTweet)

graph.add_node("optimize",optimizeTweet)


graph.add_edge(START,'generate')

graph.add_edge('generate','evaluate')


graph.add_conditional_edges('evaluate',routeEvaluation,{"approved":END,'Needs_Improvement':'optimize'})

graph.add_edge('optimize','evaluate')

workflow=graph.compile()

workflow


In [ ]:
initial_state = {
    "topic": "testing......",
    "iteration": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)

In [ ]:
result